In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/local/opt/openjdk@17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [11]:
%mkdir data
%cd data
!curl -L -O https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

/Users/jaimesolis/de-zoomcamp/de-zoomcamp/06-batch/homework/data
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 67.8M  100 67.8M    0     0  4601k      0  0:00:15  0:00:15 --:--:-- 5570k


In [7]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

'4.1.1'

## Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

What's the output?

In [5]:
spark.version

'4.1.1'

## Question 2: Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

- 6MB
- **25MB**
- 75MB
- 100MB

In [18]:
df = spark.read.parquet("data/yellow_tripdata_2025-11.parquet")

df.repartition(4) \
  .write \
  .parquet("data/pq/", mode="overwrite")

In [27]:
%ls -l pq

total 205312
-rw-r--r--  1 jaimesolis  staff         0 Mar  2 19:02 _SUCCESS
-rw-r--r--  1 jaimesolis  staff  25597759 Mar  2 19:02 part-00000-41ca608f-8933-43a3-86dc-63e069eb8e11-c000.snappy.parquet
-rw-r--r--  1 jaimesolis  staff  25583585 Mar  2 19:02 part-00001-41ca608f-8933-43a3-86dc-63e069eb8e11-c000.snappy.parquet
-rw-r--r--  1 jaimesolis  staff  25611777 Mar  2 19:02 part-00002-41ca608f-8933-43a3-86dc-63e069eb8e11-c000.snappy.parquet
-rw-r--r--  1 jaimesolis  staff  25616300 Mar  2 19:02 part-00003-41ca608f-8933-43a3-86dc-63e069eb8e11-c000.snappy.parquet


## Question 3: Count records

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

- 62,610
- 102,340
- **162,604**
- 225,768


In [29]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [32]:
from pyspark.sql import functions as F
df3 = df.filter(F.col("tpep_pickup_datetime").between("2025-11-15 00:00:00", "2025-11-15 23:59:59"))
df3.count()

162604

## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

- 22.7
- 58.2
- **90.6**
- 134.5

In [39]:
df.withColumn("trip_duration", F.col("tpep_dropoff_datetime")- F.col("tpep_pickup_datetime") ) \
    .select("tpep_pickup_datetime","tpep_dropoff_datetime", "trip_duration" ) \
    .show(truncate=False)

+--------------------+---------------------+-----------------------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration                      |
+--------------------+---------------------+-----------------------------------+
|2025-11-01 00:13:25 |2025-11-01 00:13:25  |INTERVAL '0 00:00:00' DAY TO SECOND|
|2025-11-01 00:49:07 |2025-11-01 01:01:22  |INTERVAL '0 00:12:15' DAY TO SECOND|
|2025-11-01 00:07:19 |2025-11-01 00:20:41  |INTERVAL '0 00:13:22' DAY TO SECOND|
|2025-11-01 00:00:00 |2025-11-01 01:01:03  |INTERVAL '0 01:01:03' DAY TO SECOND|
|2025-11-01 00:18:50 |2025-11-01 00:49:32  |INTERVAL '0 00:30:42' DAY TO SECOND|
|2025-11-01 00:21:11 |2025-11-01 00:31:39  |INTERVAL '0 00:10:28' DAY TO SECOND|
|2025-11-01 00:07:31 |2025-11-01 00:25:44  |INTERVAL '0 00:18:13' DAY TO SECOND|
|2025-11-01 00:46:52 |2025-11-01 01:38:55  |INTERVAL '0 00:52:03' DAY TO SECOND|
|2025-11-01 00:56:59 |2025-11-01 01:02:05  |INTERVAL '0 00:05:06' DAY TO SECOND|
|2025-11-01 00:10:43 |2025-1

In [47]:
df.withColumn("trip_duration_hours", (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600.0)\
    .select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_duration_hours")\
    .orderBy(F.desc("trip_duration_hours")) \
    .show(truncate=False)

[Stage 17:=============================>                            (4 + 4) / 8]

+--------------------+---------------------+-------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_hours|
+--------------------+---------------------+-------------------+
|2025-11-26 20:22:12 |2025-11-30 15:01:00  |90.64666666666666  |
|2025-11-27 04:22:41 |2025-11-30 09:19:35  |76.94833333333334  |
|2025-11-03 10:42:55 |2025-11-06 14:55:45  |76.21388888888889  |
|2025-11-07 11:23:22 |2025-11-10 08:40:41  |69.28861111111111  |
|2025-11-18 17:12:47 |2025-11-21 12:17:37  |67.08055555555555  |
|2025-11-22 17:45:30 |2025-11-25 09:07:36  |63.36833333333333  |
|2025-11-01 07:44:57 |2025-11-03 16:07:53  |56.382222222222225 |
|2025-11-27 14:34:56 |2025-11-29 14:43:48  |48.147777777777776 |
|2025-11-01 14:55:34 |2025-11-03 14:24:16  |47.47833333333333  |
|2025-11-22 16:05:20 |2025-11-24 13:31:59  |45.44416666666667  |
|2025-11-26 13:20:12 |2025-11-28 09:22:48  |44.04333333333334  |
|2025-11-27 15:03:04 |2025-11-29 10:16:54  |43.230555555555554 |
|2025-11-05 14:49:55 |202

## Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

- 80
- 443
- **4040**
- 8080

## Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

- **Governor's Island/Ellis Island/Liberty Island**
- Arden Heights
- Rikers Island
- Jamaica Bay

If multiple answers are correct, select any

In [49]:
!curl -L -O https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 12331  100 12331    0     0  34126      0 --:--:-- --:--:-- --:--:-- 34157


In [50]:
ls

pq/                              yellow_tripdata_2025-11.parquet
taxi_zone_lookup.csv


In [53]:
df_zones = spark.read.option("header", "true").csv("data/taxi_zone_lookup.csv")

In [56]:
df_zones.printSchema()

root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [60]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [54]:
df.createOrReplaceTempView("yellow")
df_zones.createOrReplaceTempView("zones")

In [65]:
spark.sql("""
        SELECT z.Zone, count(*) as Trips
        FROM yellow y
        JOIN zones z on y.PULocationID=z.LocationID
        Group BY 1
        Order By count(*) 

"""    
).show(truncate=False)

+---------------------------------------------+-----+
|Zone                                         |Trips|
+---------------------------------------------+-----+
|Governor's Island/Ellis Island/Liberty Island|1    |
|Eltingville/Annadale/Prince's Bay            |1    |
|Arden Heights                                |1    |
|Port Richmond                                |3    |
|Rikers Island                                |4    |
|Rossville/Woodrow                            |4    |
|Green-Wood Cemetery                          |4    |
|Great Kills                                  |4    |
|Jamaica Bay                                  |5    |
|Westerleigh                                  |12   |
|Crotona Park                                 |14   |
|Oakwood                                      |14   |
|New Dorp/Midland Beach                       |14   |
|West Brighton                                |14   |
|Willets Point                                |15   |
|Breezy Point/Fort Tilden/Ri

In [66]:
spark.stop()